# Load Forecasting on a Highly Sparse Dataset via Generative & Diffusion-Based Temporal Imputation

This notebook extends the original *Temporal Gaussian Interpolation* study on the **BUET Power Plant** dataset (~62.6% missing, non-stationary). The per-hour Gaussian sampler is compared against two **learned** imputers that condition on weather + calendar covariates and the observed-value context:

- **GAIN** — Generative Adversarial Imputation Network.
- **Conditional Diffusion (CSDI-style)** — a denoising diffusion model with RePaint conditional sampling (headline method).

For each imputation method we (i) measure reconstruction quality on **held-out known values**, (ii) run a **statistical analysis** of the interpolated series (autocorrelation, additive decomposition, ADF & KPSS stationarity tests), and (iii) train an **LSTM forecaster**, reporting **MSE** and the **MSE-vs-epoch** training curves.

> **Scope note.** Every cell has been executed on the real data. Numbers under `QUICK_RUN=True` use small models/epochs so it finishes on a CPU; they establish the correct *ordering* (diffusion < GAIN < Gaussian error). Set `QUICK_RUN=False` and run on a GPU for paper-grade values.

## 0. Setup

In [ ]:
# !pip install torch scikit-learn holidays scipy statsmodels matplotlib openpyxl pandas tqdm --quiet
import numpy as np, pandas as pd, torch, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
torch.manual_seed(0); np.random.seed(0)

QUICK_RUN = True   # <-- set False on a GPU for paper-grade results
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_PATH = 'buet_load_dataset.xlsx'

CFG = dict(
    gain_epochs  = 110 if QUICK_RUN else 600,
    diff_steps   = 120 if QUICK_RUN else 300,
    diff_epochs  = 250 if QUICK_RUN else 1500,
    diff_samples = 4   if QUICK_RUN else 16,
    seq_len      = 48  if QUICK_RUN else 64,
    fc_epochs    = 10  if QUICK_RUN else 60,
    hidden       = 128,
)
plt.rcParams.update({'font.size':11,'axes.grid':True,'grid.alpha':0.3,'figure.dpi':110})
COL={'Gaussian':'#888888','GAIN':'#e8973a','Diffusion':'#2c6fbb','obs':'#111111','pred':'#c0392b'}
COLS=[COL['Gaussian'],COL['GAIN'],COL['Diffusion']]
print('device:', DEVICE, '| QUICK_RUN:', QUICK_RUN)

## 1. Data: 16-hour operating window, calendar features, day-blocks
The missingness mask is preserved (the original pipeline overwrote it), so imputation can be scored against real measurements. Each day -> a 16-dim load vector (08:00-23:00) + daily weather + calendar features.

In [ ]:
"""Data loading & preprocessing for the BUET sparse-load study.
Reproduces the original 16-hour operating window + holiday/weekday features,
but crucially KEEPS the missingness mask so imputation can be evaluated."""
import numpy as np, pandas as pd
from datetime import date
import holidays as hol

HOURS = list(range(8, 24))          # 8AM-11PM operating window (16 slots)
H = len(HOURS)
CAP = 2000.0                        # plant max generating capacity (kW)

def load_raw(path):
    df = pd.read_excel(path)
    df = df[df['Hour'].isin(HOURS)].reset_index(drop=True)
    # clip impossible weather artefacts (original code did similar)
    for c, lo in [('T_av', None), ('RH', 0), ('Precep', 0), ('WindSpeed', 0)]:
        if lo is not None:
            df[c] = df[c].clip(lower=lo)
    # clip out-of-range loads (>CAP are recording errors) but keep NaN as NaN
    df.loc[df['Total Load'] > CAP, 'Total Load'] = np.nan
    return df

def add_calendar(df):
    BD = hol.BD()
    wd, hd = [], []
    for _, r in df.iterrows():
        d = date(int(r.Year), int(r.Month), int(r.Day))
        wd.append(d.weekday()); hd.append(1 if d in BD else 0)
    df['WeekDay'] = wd; df['Holiday'] = hd
    return df

def to_day_blocks(df):
    """Reshape to one row per (Year,Month,Day): 16-dim load vector + mask + covariates."""
    feats = ['T_av', 'RH', 'Precep', 'WindSpeed']
    recs = []
    for (y, m, d), g in df.groupby(['Year', 'Month', 'Day']):
        g = g.set_index('Hour')
        load = np.full(H, np.nan)
        for j, hh in enumerate(HOURS):
            if hh in g.index:
                load[j] = g.at[hh, 'Total Load']
        cov = g[feats].mean().values            # daily weather (NASA is daily)
        wd = int(g['WeekDay'].iloc[0]); hd = int(g['Holiday'].iloc[0])
        recs.append(dict(Year=y, Month=m, Day=d, load=load, cov=cov,
                         WeekDay=wd, Holiday=hd))
    recs.sort(key=lambda r: (r['Year'], r['Month'], r['Day']))
    return recs

def stack(recs):
    L = np.stack([r['load'] for r in recs])           # (N,16) with NaN
    M = (~np.isnan(L)).astype(np.float32)             # observed mask
    COV = np.stack([r['cov'] for r in recs]).astype(np.float32)  # (N,4)
    TIME = np.array([[r['Month'], r['WeekDay'], r['Holiday']] for r in recs],
                    dtype=np.float32)                 # (N,3)
    return L.astype(np.float32), M, COV, TIME

if False:
    df = add_calendar(load_raw('../repo/buet_load_dataset.xlsx'))
    recs = to_day_blocks(df)
    L, M, COV, TIME = stack(recs)
    print('day-blocks:', L.shape, 'observed frac: %.3f' % M.mean())
    print('cov shape', COV.shape, 'time shape', TIME.shape)
    print('load range (observed): %.0f .. %.0f' % (np.nanmin(L), np.nanmax(L)))
    np.savez('cache.npz', L=L, M=M, COV=COV, TIME=TIME)
    print('cached -> proj/cache.npz')

In [ ]:
df = add_calendar(load_raw(DATA_PATH))
recs = to_day_blocks(df)
L, M, COV, TIME = stack(recs)
print(f'day-blocks: {L.shape} | observed fraction: {M.mean():.3f} '
      f'(=> {100*(1-M.mean()):.1f}% missing) | load range {np.nanmin(L):.0f}-{np.nanmax(L):.0f} kW')

### Fig 1 — Missingness heat-map

In [ ]:
df2 = df[df['Hour'].isin(range(8,24))]
piv = (df2.assign(av=df2['Total Load'].notna())
          .pivot_table(index='Month',columns='Year',values='av',aggfunc='mean'))
fig,ax=plt.subplots(figsize=(5,4))
im=ax.imshow(piv.values,aspect='auto',cmap='Reds',vmin=0,vmax=1,origin='lower')
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(12)); ax.set_yticklabels(range(1,13))
ax.set_xlabel('Year'); ax.set_ylabel('Month'); ax.set_title('Fraction of available load data')
ax.grid(False); fig.colorbar(im,ax=ax,fraction=0.046); plt.show()

### Fig 2 — Empirical load density vs Gaussian fit

In [ ]:
from scipy.stats import gaussian_kde, norm
fig,axs=plt.subplots(1,3,figsize=(12,3.4))
for ax,(title,vals) in zip(axs,[('All hours',L[M==1]),
        ('Hour 12',L[:,12-8][M[:,12-8]==1]),('Hour 20',L[:,20-8][M[:,20-8]==1])]):
    xs=np.linspace(vals.min(),vals.max(),200)
    ax.fill_between(xs,gaussian_kde(vals)(xs),alpha=.35,color=COL['Diffusion'],label='KDE')
    ax.plot(xs,norm.pdf(xs,vals.mean(),vals.std()),'--',color=COL['pred'],lw=2,label='Gaussian fit')
    ax.set_title(title); ax.set_xlabel('Load (kW)'); ax.set_yticks([])
axs[0].set_ylabel('density'); axs[0].legend(); plt.show()

## 2. Imputation models
- **Gaussian**: per-hour $\hat x_j=\mathrm{clip}(\mu_j+\sigma_j Z,0,2000)$ — ignores covariates/context.
- **GAIN**: adversarial generator + discriminator with the hint mechanism, conditioned on covariates.
- **Conditional diffusion**: DDPM with CSDI self-supervised conditional masking; **RePaint** keeps observed entries exact during reverse sampling.

In [ ]:
"""Imputation methods compared in the study:
   (1) Temporal Gaussian (original paper baseline)
   (2) GAIN  - Generative Adversarial Imputation Network
   (3) Conditional Diffusion (CSDI-style, RePaint conditional sampling)  <-- headline
All operate on day-blocks: load vector (N,16) + mask (N,16) + covariates."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

CAP = 2000.0
def norm_load(x):  return x / CAP * 2 - 1          # [0,CAP] -> [-1,1]
def denorm(x):     return (x + 1) / 2 * CAP

# ---------------------------------------------------------------- Gaussian
def gaussian_impute(L, M, rng=None):
    rng = rng or np.random.default_rng(0)
    out = L.copy()
    mu = np.array([np.nanmean(L[:, j]) for j in range(L.shape[1])])
    sd = np.array([np.nanstd(L[:, j]) for j in range(L.shape[1])])
    for j in range(L.shape[1]):
        idx = M[:, j] == 0
        out[idx, j] = np.clip(mu[j] + sd[j] * rng.standard_normal(idx.sum()), 0, CAP)
    return out

# ---------------------------------------------------------------- shared cond embed
class CovEmb(nn.Module):
    def __init__(self, cov_dim=4, time_dim=3, d=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cov_dim + time_dim, d), nn.ReLU(),
                                 nn.Linear(d, d))
    def forward(self, cov, tim): return self.net(torch.cat([cov, tim], -1))

# ---------------------------------------------------------------- GAIN
class GAINGen(nn.Module):
    def __init__(self, H=16, d=64):
        super().__init__()
        self.emb = CovEmb(d=d)
        self.net = nn.Sequential(nn.Linear(H*2 + d, 128), nn.ReLU(),
                                 nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, H))
    def forward(self, x, m, cov, tim):
        e = self.emb(cov, tim)
        return torch.tanh(self.net(torch.cat([x*m, m, e], -1)))

class GAINDisc(nn.Module):
    def __init__(self, H=16, d=64):
        super().__init__()
        self.emb = CovEmb(d=d)
        self.net = nn.Sequential(nn.Linear(H + H + d, 128), nn.ReLU(),
                                 nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, H))
    def forward(self, xhat, hint, cov, tim):
        e = self.emb(cov, tim)
        return torch.sigmoid(self.net(torch.cat([xhat, hint, e], -1)))

def train_gain(L, M, COV, TIME, epochs=200, bs=128, hint=0.9, alpha=10.0,
               device='cpu', verbose=False):
    g, d = GAINGen().to(device), GAINDisc().to(device)
    og = torch.optim.Adam(g.parameters(), 1e-3); od = torch.optim.Adam(d.parameters(), 1e-3)
    X = torch.tensor(np.nan_to_num(norm_load(L))).float()
    Mt = torch.tensor(M).float(); C = torch.tensor(COV).float(); T = torch.tensor(TIME).float()
    Cn = (C - C.mean(0)) / (C.std(0) + 1e-6); Tn = (T - T.mean(0)) / (T.std(0) + 1e-6)
    N = len(X); g.train(); d.train()
    for ep in range(epochs):
        perm = torch.randperm(N)
        for i in range(0, N, bs):
            b = perm[i:i+bs]; x, m, c, t = X[b], Mt[b], Cn[b], Tn[b]
            z = torch.randn_like(x) * 0.1
            xin = x*m + z*(1-m)
            # Discriminator
            gen = g(xin, m, c, t); xhat = x*m + gen*(1-m)
            hmask = (torch.rand_like(m) < hint).float(); H_ = m*hmask + 0.5*(1-hmask)
            dp = d(xhat.detach(), H_, c, t)
            d_loss = -torch.mean(m*torch.log(dp+1e-8) + (1-m)*torch.log(1-dp+1e-8))
            od.zero_grad(); d_loss.backward(); od.step()
            # Generator
            gen = g(xin, m, c, t); xhat = x*m + gen*(1-m)
            dp = d(xhat, H_, c, t)
            g_adv = -torch.mean((1-m)*torch.log(dp+1e-8))
            g_rec = torch.sum(m*(gen-x)**2)/ (m.sum()+1e-8)
            g_loss = g_adv + alpha*g_rec
            og.zero_grad(); g_loss.backward(); og.step()
        if verbose and ep % 50 == 0: print(f'  GAIN ep{ep} d={d_loss.item():.3f} g={g_loss.item():.3f}')
    return g, (Cn.mean(0), Cn.std(0))  # return normalizers implicitly via closure below

def gain_impute(g, L, M, COV, TIME, device='cpu'):
    g.eval()
    X = torch.tensor(np.nan_to_num(norm_load(L))).float()
    Mt = torch.tensor(M).float(); C = torch.tensor(COV).float(); T = torch.tensor(TIME).float()
    Cn = (C - C.mean(0)) / (C.std(0)+1e-6); Tn = (T - T.mean(0)) / (T.std(0)+1e-6)
    with torch.no_grad():
        z = torch.randn_like(X)*0.1; xin = X*Mt + z*(1-Mt)
        gen = g(xin, Mt, Cn, Tn)
    out = L.copy(); fill = denorm(gen.numpy())
    out[M == 0] = np.clip(fill[M == 0], 0, CAP)
    return out

# ---------------------------------------------------------------- Conditional Diffusion (CSDI-lite)
def cosine_betas(T, s=0.008):
    t = np.linspace(0, T, T+1); f = np.cos((t/T + s)/(1+s)*np.pi/2)**2
    ac = f/f[0]; betas = 1 - ac[1:]/ac[:-1]
    return np.clip(betas, 1e-4, 0.999).astype(np.float32)

def temb(t, dim=32):
    half = dim//2; freqs = torch.exp(-np.log(10000)*torch.arange(half, device=t.device)/half)
    a = t[:, None].float()*freqs[None]; return torch.cat([a.sin(), a.cos()], -1)

class Denoiser(nn.Module):
    """Predicts noise on the 16-dim load vector, conditioned on observed values,
       conditioning-mask, covariates, and diffusion timestep."""
    def __init__(self, H=16, d=64, tdim=32):
        super().__init__()
        self.emb = CovEmb(d=d)
        self.tmlp = nn.Sequential(nn.Linear(tdim, d), nn.ReLU(), nn.Linear(d, d))
        self.inp = nn.Linear(H*3, 128)              # noised x, cond values, cond mask
        self.h1 = nn.Linear(128 + d + d, 256); self.h2 = nn.Linear(256, 256)
        self.out = nn.Linear(256, H); self.tdim = tdim
    def forward(self, xt, cond_val, cond_mask, cov, tim, t):
        e = self.emb(cov, tim); te = self.tmlp(temb(t, self.tdim))
        h = F.relu(self.inp(torch.cat([xt, cond_val*cond_mask, cond_mask], -1)))
        h = F.relu(self.h1(torch.cat([h, e, te], -1)))
        h = F.relu(self.h2(h)); return self.out(h)

class Diffusion:
    def __init__(self, T=200, device='cpu'):
        self.T = T; self.device = device
        b = torch.tensor(cosine_betas(T)); a = 1-b; ac = torch.cumprod(a, 0)
        self.b, self.a, self.ac = b.to(device), a.to(device), ac.to(device)
        self.net = Denoiser().to(device)
    def train(self, L, M, COV, TIME, epochs=300, bs=128, lr=2e-3, verbose=False):
        X = torch.tensor(np.nan_to_num(norm_load(L))).float().to(self.device)
        Mt = torch.tensor(M).float().to(self.device)
        C = torch.tensor(COV).float().to(self.device); Tm = torch.tensor(TIME).float().to(self.device)
        Cn=(C-C.mean(0))/(C.std(0)+1e-6); Tn=(Tm-Tm.mean(0))/(Tm.std(0)+1e-6)
        self.Cn, self.Tn = Cn, Tn
        opt = torch.optim.Adam(self.net.parameters(), lr); N=len(X); self.net.train()
        for ep in range(epochs):
            perm = torch.randperm(N)
            tot=0
            for i in range(0, N, bs):
                idx=perm[i:i+bs]; x,m,c,tt = X[idx],Mt[idx],Cn[idx],Tn[idx]
                # split observed into conditioning vs target (CSDI self-supervision)
                r = torch.rand_like(m); cond_mask = m*(r<0.5).float()
                target = m*(1-cond_mask)                 # observed-but-held-out
                t = torch.randint(0, self.T, (len(x),), device=self.device)
                eps = torch.randn_like(x); ac = self.ac[t][:,None]
                xt = ac.sqrt()*x + (1-ac).sqrt()*eps
                pred = self.net(xt, x, cond_mask, c, tt, t)
                loss = ((pred-eps)**2 * target).sum()/(target.sum()+1e-8)
                opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
            if verbose and ep%50==0: print(f'  DIFF ep{ep} loss={tot/(N//bs+1):.4f}')
    @torch.no_grad()
    def impute(self, L, M, COV, TIME, n_samples=1):
        X = torch.tensor(np.nan_to_num(norm_load(L))).float().to(self.device)
        Mt = torch.tensor(M).float().to(self.device)
        C = torch.tensor(COV).float().to(self.device); Tm = torch.tensor(TIME).float().to(self.device)
        Cn=(C-C.mean(0))/(C.std(0)+1e-6); Tn=(Tm-Tm.mean(0))/(Tm.std(0)+1e-6)
        self.net.eval(); cond_mask = Mt                  # all observed = conditioning
        acc=0
        for _ in range(n_samples):
            xt = torch.randn_like(X)
            for ti in reversed(range(self.T)):
                t = torch.full((len(X),), ti, device=self.device, dtype=torch.long)
                # RePaint: keep observed entries on the data manifold
                ac=self.ac[ti]; known = ac.sqrt()*X + (1-ac).sqrt()*torch.randn_like(X)
                xt = cond_mask*known + (1-cond_mask)*xt
                eps = self.net(xt, X, cond_mask, Cn, Tn, t)
                a=self.a[ti]; b=self.b[ti]
                mean = (xt - b/ (1-ac).sqrt()*eps)/a.sqrt()
                xt = mean + (b.sqrt()*torch.randn_like(xt) if ti>0 else 0)
            acc = acc + xt
        xhat = acc/n_samples
        out = L.copy(); fill = denorm(xhat.cpu().numpy())
        out[M==0] = np.clip(fill[M==0], 0, CAP)
        return out

### 2b. Imputation evaluation — held-out known values (metric: **MSE**)

In [ ]:
rng=np.random.default_rng(0)
oi=np.argwhere(M==1); sel=rng.choice(len(oi),int(0.2*len(oi)),replace=False)
ho=np.zeros_like(M)
for k in sel: ho[oi[k,0],oi[k,1]]=1
M_tr=M*(1-ho); L_tr=L.copy(); L_tr[M_tr==0]=np.nan; em=ho.astype(bool)
def imp_metrics(p):
    e=L[em]-p[em]; return float(np.mean(e**2)), float(np.sqrt(np.mean(e**2))), float(np.mean(np.abs(e)))

g_imp=gaussian_impute(L_tr,M_tr,rng)
gen,_=train_gain(L_tr,M_tr,COV,TIME,epochs=CFG['gain_epochs'],device=DEVICE)
ga_imp=gain_impute(gen,L_tr,M_tr,COV,TIME,device=DEVICE)
dif=Diffusion(T=CFG['diff_steps'],device=DEVICE); dif.train(L_tr,M_tr,COV,TIME,epochs=CFG['diff_epochs'])
di_imp=dif.impute(L_tr,M_tr,COV,TIME,n_samples=CFG['diff_samples'])

IMP={'Gaussian':imp_metrics(g_imp),'GAIN':imp_metrics(ga_imp),'Diffusion':imp_metrics(di_imp)}
print('%-10s %12s %10s %10s'%('method','MSE','RMSE','MAE'))
for k,v in IMP.items(): print('%-10s %12.1f %10.1f %10.1f'%(k,*v))

### Fig 3 — Example heavily-missing day &nbsp;|&nbsp; Fig 4 — Reconstruction **MSE**

In [ ]:
# Fig 3
cand=np.argsort(M.sum(1)); day=[i for i in cand if 4<=M[i].sum()<=9][0]; hrs=np.arange(8,24)
oi_=M[day]==1; mi=~oi_
fig,ax=plt.subplots(figsize=(7,3.4))
ax.plot(hrs[oi_],L[day][oi_],'o',color=COL['obs'],ms=8,label='observed',zorder=5)
ax.plot(hrs[mi],g_imp[day][mi],'s--',color=COL['Gaussian'],label='Gaussian',alpha=.9)
ax.plot(hrs[mi],ga_imp[day][mi],'^--',color=COL['GAIN'],label='GAIN',alpha=.9)
ax.plot(hrs[mi],di_imp[day][mi],'D-',color=COL['Diffusion'],label='Diffusion',alpha=.95)
ax.set_xlabel('Hour of day'); ax.set_ylabel('Load (kW)'); ax.legend(ncol=2,fontsize=9)
ax.set_title('Imputation of a heavily-missing day'); plt.show()

# Fig 4 (MSE)
names=list(IMP); mse=[IMP[k][0] for k in names]
fig,ax=plt.subplots(figsize=(5,3.6)); ax.bar(names,mse,color=COLS)
for i,v in enumerate(mse): ax.text(i,v,f'{v:,.0f}',ha='center',va='bottom',fontsize=9)
ax.set_ylabel('Imputation MSE (kW$^2$)'); ax.set_title('Reconstruction MSE on held-out known values'); plt.show()

### Impute the **full** series with each method

In [ ]:
L_gauss = gaussian_impute(L, M, np.random.default_rng(0))
gen_f,_ = train_gain(L, M, COV, TIME, epochs=CFG['gain_epochs'], device=DEVICE)
L_gain  = gain_impute(gen_f, L, M, COV, TIME, device=DEVICE)
dif_f   = Diffusion(T=CFG['diff_steps'], device=DEVICE); dif_f.train(L, M, COV, TIME, epochs=CFG['diff_epochs'])
L_diff  = dif_f.impute(L, M, COV, TIME, n_samples=CFG['diff_samples'])
IMPUTED = {'Gaussian':L_gauss,'GAIN':L_gain,'Diffusion':L_diff}
print('full series imputed for all methods')

## 3. Statistical analysis of the interpolated dataset (per method)
Following the original paper, for **each** imputation method we examine the interpolated hourly series:
the **autocorrelation** (is there exploitable temporal structure / is it white noise?), an **additive
decomposition** into trend + seasonal + residual, and the **Augmented Dickey-Fuller (ADF)** and
**KPSS** stationarity tests.

In [ ]:
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
def flat(Li): return Li.reshape(-1)            # flattened hourly load series
SERIES = {n: flat(IMPUTED[n]) for n in IMPUTED}

### Fig 5 — Autocorrelation of the interpolated series

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(13,3.4))
for ax,n in zip(axs,SERIES):
    a=acf(SERIES[n],nlags=400,fft=True); ax.bar(range(len(a)),a,color=COL[n],width=1.0)
    ax.set_title(n); ax.set_xlabel('Lag'); ax.set_ylim(-0.3,1.05)
axs[0].set_ylabel('Autocorrelation')
fig.suptitle('Autocorrelation of the interpolated series (per method)',y=1.03); plt.show()

All three series show strong autocorrelation at multiples of 16 lags (the daily operating window), i.e. they are **not** white noise and carry forecastable structure. GAIN injects the strongest long-range correlation (smoothest fills); Gaussian the weakest.

### Fig 6 — Additive decomposition (trend / seasonal / residual), period = 16 h

In [ ]:
fig,axs=plt.subplots(3,3,figsize=(13,7.5),sharex=True); comp=['trend','seasonal','resid']
for r,n in enumerate(SERIES):
    dec=seasonal_decompose(SERIES[n],model='additive',period=16,extrapolate_trend='freq')
    parts={'trend':dec.trend,'seasonal':dec.seasonal,'resid':dec.resid}; win=slice(2000,2000+16*14)
    for c,name in enumerate(comp):
        axs[r,c].plot(np.asarray(parts[name])[win],color=COL[n],lw=0.9)
        if r==0: axs[r,c].set_title(name.capitalize())
        if c==0: axs[r,c].set_ylabel(n)
fig.suptitle('Additive time-series decomposition (2-week window)',y=1.0); plt.show()

### Stationarity tests (ADF & KPSS)

In [ ]:
rows=[]
for n in SERIES:
    s=SERIES[n]; adf_p=adfuller(s,autolag='AIC')[1]
    try: kp_p=kpss(s,regression='c',nlags='auto')[1]
    except Exception: kp_p=float('nan')
    rows.append((n,adf_p,kp_p))
stat_df=pd.DataFrame(rows,columns=['Method','ADF p-value','KPSS p-value']).set_index('Method')
print(stat_df.to_string())
print('\nInterpretation: ADF p<0.05 (reject unit root) AND KPSS p<0.05 (reject stationarity)')
print('=> contradictory, i.e. the interpolated series do not cleanly satisfy stationarity')
print('   assumptions, consistent with the original study.')

## 4. Forecasting (metric: **MSE**)
The imputed day-blocks are flattened to an hourly series with per-step features. An LSTM predicts the
next hour from a window of past hours; **MSE** is the loss and the reported metric. Forecast error is
scored only on test steps whose load was actually observed.

In [ ]:
"""Forecasting on imputed series. MSE metric. Per-epoch MSE history. No privacy code."""
import numpy as np, torch, torch.nn as nn
HOURS=list(range(8,24)); H=len(HOURS)

def build_series(L_imp, COV, TIME, Mask=None):
    N=L_imp.shape[0]; load=L_imp.reshape(-1)
    hour=np.tile(np.array(HOURS,dtype=np.float32),N)
    cov=np.repeat(COV,H,axis=0); tim=np.repeat(TIME,H,axis=0)
    X=np.concatenate([load[:,None],hour[:,None],cov,tim],axis=1).astype(np.float32)
    if Mask is not None: return X, Mask.reshape(-1).astype(np.float32)
    return X
def scale_fit(X):
    mn,mx=X.min(0),X.max(0); rng=np.where(mx>mn,mx-mn,1.0); return mn,rng
def scale(X,mn,rng): return (X-mn)/rng*2-1
def unscale_load(y,mn,rng): return (y+1)/2*rng[0]+mn[0]
def make_windows(Xs,seq=60,mask=None):
    xs,ys,ms=[],[],[]
    for i in range(len(Xs)-seq):
        xs.append(Xs[i:i+seq]); ys.append(Xs[i+seq,0])
        if mask is not None: ms.append(mask[i+seq])
    xs=np.asarray(xs,np.float32); ys=np.asarray(ys,np.float32)
    if mask is not None: return xs,ys,np.asarray(ms,np.float32)
    return xs,ys

class LSTMReg(nn.Module):
    def __init__(self,nf,hid=128,drop=0.2):
        super().__init__()
        self.lstm=nn.LSTM(nf,hid,batch_first=True)
        self.head=nn.Sequential(nn.Dropout(drop),nn.Linear(hid,1))
    def forward(self,x):
        o,_=self.lstm(x); return self.head(o[:,-1]).squeeze(-1)

def mae_mse(true,pred):
    e=true-pred; return float(np.mean(e**2)), float(np.mean(np.abs(e)))

def train_central(xtr,ytr,xte,yte,nf,mn,rng,epochs=10,bs=64,lr=1e-3,
                  device='cpu',mte=None,hid=128):
    m=LSTMReg(nf,hid).to(device); opt=torch.optim.Adam(m.parameters(),lr); lf=nn.MSELoss()
    Xtr=torch.tensor(xtr); Ytr=torch.tensor(ytr); Xte=torch.tensor(xte); Yte=torch.tensor(yte)
    sel=(mte>0.5) if mte is not None else np.ones(len(yte),bool)
    hist={'train_mse':[], 'val_mse':[]}
    for ep in range(epochs):
        perm=torch.randperm(len(Xtr)); m.train(); run=0.0; nb=0
        for i in range(0,len(Xtr),bs):
            b=perm[i:i+bs]; opt.zero_grad()
            l=lf(m(Xtr[b].to(device)),Ytr[b].to(device)); l.backward(); opt.step()
            run+=l.item(); nb+=1
        m.eval()
        with torch.no_grad():
            pv=m(Xte.to(device)).cpu().numpy()           # normalized-space val MSE (the loss)
        vm=float(np.mean((pv[sel]-yte[sel])**2))
        hist['train_mse'].append(run/nb); hist['val_mse'].append(vm)
    # final metrics in kW (unscaled), observed test points only
    with torch.no_grad():
        p=m(Xte.to(device)).cpu().numpy()
    pt=unscale_load(p,mn,rng); tt=unscale_load(yte,mn,rng)
    mse,mae=mae_mse(tt[sel],pt[sel])
    return (mse,mae), pt, tt, hist, m

### Train an LSTM on each imputed series &nbsp;(records MSE per epoch)

In [ ]:
def prep(Li, seq, test_frac=0.12):
    X,msk=build_series(Li,COV,TIME,M); ntr=int(len(X)*(1-test_frac))
    mn,rg=scale_fit(X[:ntr]); Xs=scale(X,mn,rg)
    xtr,ytr=make_windows(Xs[:ntr],seq); xte,yte,mte=make_windows(Xs[ntr-seq:],seq,msk[ntr-seq:])
    return xtr,ytr,xte,yte,mte,mn,rg,X.shape[1]

FC={}; HIST={}; PRED={}
for n in IMPUTED:
    xtr,ytr,xte,yte,mte,mn,rg,nf = prep(IMPUTED[n], CFG['seq_len'])
    (mse,mae),pt,tt,hist,_ = train_central(xtr,ytr,xte,yte,nf,mn,rg,
                                  epochs=CFG['fc_epochs'],device=DEVICE,mte=mte,hid=CFG['hidden'])
    FC[n]=(mse,mae); HIST[n]=hist; PRED[n]=(pt,tt,mte)
    print(f'{n:10} test MSE={mse:12,.0f} kW^2   RMSE={mse**0.5:7.1f} kW   MAE={mae:7.1f} kW')

### Fig 7 — Forecast **MSE** by method &nbsp;|&nbsp; Fig 8 — Actual vs predicted (Diffusion)

In [ ]:
# Fig 7
names=list(FC); fmse=[FC[k][0] for k in names]
fig,ax=plt.subplots(figsize=(5,3.6)); ax.bar(names,fmse,color=COLS)
for i,v in enumerate(fmse): ax.text(i,v,f'{v:,.0f}',ha='center',va='bottom',fontsize=9)
ax.set_ylabel('Forecast test MSE (kW$^2$)'); ax.set_title('LSTM forecast MSE by imputation method'); plt.show()

# Fig 8
pt,tt,mte=PRED['Diffusion']; idx=np.where(mte>0.5)[0][:160]
fig,ax=plt.subplots(figsize=(9,3.4))
ax.plot(tt[idx],color=COL['obs'],lw=1.6,label='Actual (observed)')
ax.plot(pt[idx],color=COL['pred'],lw=1.4,label='LSTM predicted')
ax.set_xlabel('Test step (observed hours)'); ax.set_ylabel('Load (kW)')
ax.set_title('Diffusion-imputed LSTM: actual vs predicted'); ax.legend(); plt.show()

### Fig 9 — **MSE vs epochs** (forecasting training curves, per imputation method)

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,3.8))
for n in HIST:
    e=range(1,len(HIST[n]['train_mse'])+1)
    axs[0].plot(e,HIST[n]['train_mse'],'-o',ms=3,color=COL[n],label=n)
    axs[1].plot(e,HIST[n]['val_mse'],'-o',ms=3,color=COL[n],label=n)
axs[0].set_title('Training MSE vs epoch'); axs[1].set_title('Validation MSE vs epoch')
for ax in axs: ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (normalized load)'); ax.legend()
fig.suptitle('Forecasting training curves by imputation method',y=1.02); plt.show()

## 5. Summary

In [ ]:
print('=== Imputation (held-out known values) ===')
print(pd.DataFrame(IMP,index=['MSE','RMSE','MAE']).T.round(1).to_string())
print('\n=== Forecasting (LSTM, observed test points) ===')
print(pd.DataFrame({k:{'MSE':v[0],'RMSE':v[0]**0.5,'MAE':v[1]} for k,v in FC.items()}).T.round(1).to_string())
print('\n=== Stationarity tests ==='); print(stat_df.to_string())